# Centralized vs Decentralized UAV Evaluation

This notebook summarizes the corridor experiment results from `uav_eval/`.

It compares:

- **Centralized controller**: one main formation controller computes all UAV setpoints.
- **Decentralized baseline**: local UAV controllers use fixed coordination rules.

Expected input files:

```text
uav_eval/centralized/summary.json
uav_eval/centralized/pose.csv
uav_eval/decentralized/summary.json
uav_eval/decentralized/pose.csv
```

In [ ]:
from pathlib import Path
import csv
import json
import math

import matplotlib.pyplot as plt
from IPython.display import Markdown, display

ROOT = Path.cwd()
EVAL_DIR = ROOT / "uav_eval"
MODES = ["centralized", "decentralized"]
OFFSETS = {"F1": (2.0, 0.0, 0.0), "F2": (4.0, 0.0, 0.0)}

def load_summary(mode):
    with open(EVAL_DIR / mode / "summary.json", "r", encoding="utf-8") as fh:
        return json.load(fh)

def to_float(value):
    try:
        return float(value)
    except (TypeError, ValueError):
        return float("nan")

def load_pose(mode):
    path = EVAL_DIR / mode / "pose.csv"
    with open(path, "r", encoding="utf-8", newline="") as fh:
        rows = list(csv.DictReader(fh))
    for row in rows:
        for key, value in list(row.items()):
            if key not in ("motion", "wp_idx", "avoid"):
                row[key] = to_float(value)
        row["motion"] = int(float(row["motion"]))
        row["wp_idx"] = int(float(row["wp_idx"]))
        row["avoid"] = int(float(row["avoid"]))
    return rows

def finite_xyz(row, prefix):
    xyz = (row[f"{prefix}_x"], row[f"{prefix}_y"], row[f"{prefix}_z"])
    return all(math.isfinite(v) for v in xyz)

def xyz(row, prefix):
    return (row[f"{prefix}_x"], row[f"{prefix}_y"], row[f"{prefix}_z"])

def dist(a, b):
    return math.sqrt(sum((x - y) ** 2 for x, y in zip(a, b)))

summaries = {mode: load_summary(mode) for mode in MODES}
poses = {mode: load_pose(mode) for mode in MODES}

summaries

## Summary Table

This table is the main pass/fail comparison. A clean run should have `mission_success = True`, no failure reasons, no corridor violation, no stale odometry, and no oscillation.

In [ ]:
headers = [
    "Mode", "Success", "Mission time (s)", "Avg formation error (m)",
    "Max formation error (m)", "Corridor violation", "Stale odometry", "Oscillation", "Failure reasons"
]
lines = ["| " + " | ".join(headers) + " |", "|" + "|".join(["---"] * len(headers)) + "|"]
for mode in MODES:
    s = summaries[mode]
    lines.append(
        "| " + " | ".join([
            mode,
            str(s["mission_success"]),
            f"{s['mission_time_sec']:.2f}" if s["mission_time_sec"] is not None else "n/a",
            f"{s['formation_error_avg_m']:.3f}" if s["formation_error_avg_m"] is not None else "n/a",
            f"{s['formation_error_max_m']:.3f}" if s["formation_error_max_m"] is not None else "n/a",
            str(s["corridor_violation"]),
            str(s["stale_odometry"]),
            str(s["oscillation_detected"]),
            ", ".join(s["failure_reasons"]) if s["failure_reasons"] else "none",
        ]) + " |"
    )
display(Markdown("\n".join(lines)))

## Key Metric Comparison

These plots show the headline comparison: mission time and formation error. Lower is better for all three charts.

In [ ]:
metric_specs = [
    ("mission_time_sec", "Mission time (s)"),
    ("formation_error_avg_m", "Average formation error (m)"),
    ("formation_error_max_m", "Max formation error (m)"),
]

fig, axes = plt.subplots(1, 3, figsize=(13, 3.5))
for ax, (key, title) in zip(axes, metric_specs):
    values = [summaries[m][key] for m in MODES]
    ax.bar(MODES, values)
    ax.set_title(title)
    ax.grid(axis="y", alpha=0.3)
    for i, value in enumerate(values):
        ax.text(i, value, f"{value:.2f}", ha="center", va="bottom")
plt.tight_layout()
plt.show()

## XY Trajectory In The Corridor

This plot shows the path followed by all UAVs in the Gazebo ENU world frame. Gray rectangles are the building obstacles; the gap between the rows is the corridor.

In [ ]:
def plot_obstacles(ax):
    for cx in (12, 24, 36):
        for cy in (-6, 6):
            rect = plt.Rectangle((cx - 3, cy - 3), 6, 6, alpha=0.25)
            ax.add_patch(rect)
    ax.axhline(3, linestyle="--", linewidth=1)
    ax.axhline(-3, linestyle="--", linewidth=1)
    ax.set_aspect("equal", adjustable="box")
    ax.set_xlim(-2, 48)
    ax.set_ylim(-11, 11)
    ax.set_xlabel("x East (m)")
    ax.set_ylabel("y North (m)")
    ax.grid(alpha=0.25)

fig, axes = plt.subplots(1, 2, figsize=(14, 4.5), sharex=True, sharey=True)
for ax, mode in zip(axes, MODES):
    plot_obstacles(ax)
    for label, prefix in [("Leader", "L_odom"), ("Follower 1", "F1_odom"), ("Follower 2", "F2_odom")]:
        xs = [r[f"{prefix}_x"] for r in poses[mode] if finite_xyz(r, prefix)]
        ys = [r[f"{prefix}_y"] for r in poses[mode] if finite_xyz(r, prefix)]
        ax.plot(xs, ys, label=label)
    ax.set_title(f"{mode.capitalize()} XY path")
    ax.legend(loc="upper left")
plt.tight_layout()
plt.show()

## Formation Error Over Time

Formation error is measured as the 3D distance from each follower to its desired leader-relative offset. The dashed line is the pass/fail threshold used by the experiment (`2 m`).

In [ ]:
def formation_error_series(rows):
    times, f1_err, f2_err = [], [], []
    for r in rows:
        if not (finite_xyz(r, "L_odom") and finite_xyz(r, "F1_odom") and finite_xyz(r, "F2_odom")):
            continue
        leader = xyz(r, "L_odom")
        f1 = xyz(r, "F1_odom")
        f2 = xyz(r, "F2_odom")
        times.append(r["t_sec"])
        f1_err.append(dist(f1, (leader[0] + 2, leader[1], leader[2])))
        f2_err.append(dist(f2, (leader[0] + 4, leader[1], leader[2])))
    return times, f1_err, f2_err

fig, axes = plt.subplots(1, 2, figsize=(14, 4), sharey=True)
for ax, mode in zip(axes, MODES):
    times, f1_err, f2_err = formation_error_series(poses[mode])
    ax.plot(times, f1_err, label="Follower 1 error")
    ax.plot(times, f2_err, label="Follower 2 error")
    ax.axhline(2.0, linestyle="--", linewidth=1, label="2 m threshold")
    ax.set_title(f"{mode.capitalize()} formation error")
    ax.set_xlabel("time (s)")
    ax.set_ylabel("error (m)")
    ax.grid(alpha=0.3)
    ax.legend()
plt.tight_layout()
plt.show()

## Altitude And Lateral Deviation

The corridor is centered at `y=0` and the cruise altitude is `z=6 m`. These plots help reveal unsafe lateral/vertical excursions that may not be obvious from aggregate metrics.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 7), sharex="col")
for col, mode in enumerate(MODES):
    rows = poses[mode]
    for label, prefix in [("Leader", "L_odom"), ("Follower 1", "F1_odom"), ("Follower 2", "F2_odom")]:
        ts = [r["t_sec"] for r in rows if finite_xyz(r, prefix)]
        ys = [r[f"{prefix}_y"] for r in rows if finite_xyz(r, prefix)]
        zs = [r[f"{prefix}_z"] for r in rows if finite_xyz(r, prefix)]
        axes[0, col].plot(ts, ys, label=label)
        axes[1, col].plot(ts, zs, label=label)
    axes[0, col].axhline(2.8, linestyle="--", linewidth=1)
    axes[0, col].axhline(-2.8, linestyle="--", linewidth=1)
    axes[1, col].axhline(6.0, linestyle="--", linewidth=1)
    axes[0, col].set_title(f"{mode.capitalize()} lateral deviation")
    axes[1, col].set_title(f"{mode.capitalize()} altitude")
    axes[0, col].set_ylabel("y (m)")
    axes[1, col].set_ylabel("z (m)")
    axes[1, col].set_xlabel("time (s)")
    axes[0, col].grid(alpha=0.3)
    axes[1, col].grid(alpha=0.3)
    axes[0, col].legend()
plt.tight_layout()
plt.show()

## Interpretation For The Report

Use the summary table and plots to support these points:

- Both approaches complete the same known-map corridor scenario under the same PX4/Gazebo/ROS2 setup.
- The centralized controller is faster and has lower formation error.
- The decentralized baseline is slower and slightly less precise, but still satisfies the success threshold in the final tuned run.
- The comparison is fair because both modes use the same route, spawn positions, corridor geometry, and metric code.

For final submissions, keep the `summary.json` files as machine-readable evidence and export this notebook with plots if you need presentation figures.